### Import libraries

In [ ]:
import pandas as pd
from pandas import json_normalize

import os

In [ ]:
# Get the current working directory
current_directory = os.getcwd() 
# print(current_directory)

c:\Users\andre\Desktop\progetti\spotify wrapped


## Import and cleaning data

In [149]:
file_path = r'C:\Users\andre\Desktop\progetti\spotify wrapped\Spotify Extended Streaming History\Streaming_History_Audio_2024_23.json'
df = pd.read_json(file_path)

file_path_old = r'C:\Users\andre\Desktop\progetti\spotify wrapped\Spotify Extended Streaming History\Streaming_History_Audio_2023-2024_22.json'
df_old = pd.read_json(file_path_old)

In [150]:
df = pd.concat([df, df_old], ignore_index=True)
df.head()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2024-04-02T11:59:16Z,android,17648,IT,93.45.87.138,Maria Maria (feat. The Product G&B),Santana,Supernatural,spotify:track:3XKIUb7HzIF1Vu9usunMzc,None,None,None,clickrow,remote,False,False,False,1712058876,False
1,2024-04-02T12:05:41Z,windows,386906,IT,5.43.248.44,Money Trees,Kendrick Lamar,"good kid, m.A.A.d city",spotify:track:74tLlkN3rgVzRqQJgPfink,None,None,None,clickrow,trackdone,False,False,False,1712059154,False
2,2024-04-02T12:07:38Z,windows,114934,IT,5.43.248.44,United In Grief,Kendrick Lamar,Mr. Morale & The Big Steppers,spotify:track:5Gt9bxniM1SxN61yRzRhXL,None,None,None,trackdone,endplay,False,True,False,1712059541,False
3,2024-04-02T12:28:28Z,windows,380480,IT,5.43.248.44,Runaway,Kanye West,My Beautiful Dark Twisted Fantasy,spotify:track:3DK6m7It6Pw857FcQftMds,None,None,None,clickrow,endplay,False,True,False,1712059658,False
4,2024-04-02T12:28:47Z,windows,17420,IT,5.43.248.44,Jägermeister,Da Tweekaz,10 Years Da Tweekaz - The Definitive Collection,spotify:track:19UGEZoMAqqaTEcEYNvsox,None,None,None,playbtn,endplay,False,True,False,1712060909,False


In [151]:
# keep only the data from the year 2024, considered in my spotify wrapped

df['ts'] = pd.to_datetime(df['ts'])
df = df[df['ts'].dt.year > 2023]
df = df[df['ts'] < '2024-11-15']

In [152]:
# Count the occurrences of each song and create a new column 'listened'
df['listened'] = df['spotify_track_uri'].map(df['spotify_track_uri'].value_counts())

In [153]:
# sort the dataframe by the 'listened' column in descending order

df_sorted = df.sort_values(by='listened', ascending=False) 

In [154]:
# Group by 'spotify_track_uri' and aggregate the required information
df_grouped = df.groupby('spotify_track_uri').agg({
    'master_metadata_track_name': 'first',
    'master_metadata_album_artist_name': 'first',
    'master_metadata_album_album_name': 'first',
    'listened': 'first',
    'skipped': 'sum',
    'ms_played': 'sum',
    'reason_start': lambda x: x.value_counts().to_dict(),
    'reason_end': lambda x: x.value_counts().to_dict(),
    'incognito_mode': 'sum',
    'offline': 'sum'
}).reset_index()

df_grouped.rename(columns={'master_metadata_track_name': 'title', 'master_metadata_album_artist_name': 'artist', 'master_metadata_album_album_name' : 'album'}, inplace=True)


In [155]:
# create n columns for each reason that started a song
reason_start = set()
for reason_start_dict in df_grouped['reason_start']:
    reason_start.update(reason_start_dict.keys())
for reason in reason_start:
    df_grouped[f'reason_start_{reason}'] = df_grouped['reason_start'].apply(lambda x: x.get(reason, 0))

# create n columns for each reason that ended a song

reason_end = set()
for reason_end_dict in df_grouped['reason_end']:
    reason_end.update(reason_end_dict.keys())
for reason in reason_end:
    df_grouped[f'reason_end_{reason}'] = df_grouped['reason_end'].apply(lambda x: x.get(reason, 0))

# drop the original reason_start and reason_end columns
df_grouped.drop(columns=['reason_start', 'reason_end'], inplace=True)

In [156]:
df_grouped.head()

,spotify_track_uri,title,artist,album,listened,skipped,ms_played,incognito_mode,offline,reason_start_trackerror,...,reason_end_unexpected-exit-while-paused,reason_end_logout,reason_end_trackerror,reason_end_unknown,reason_end_endplay,reason_end_fwdbtn,reason_end_remote,reason_end_unexpected-exit,reason_end_backbtn,reason_end_trackdone
0,spotify:track:003vvx7Niy0yvhvHt4a68B,Mr. Brightside,The Killers,Hot Fuss,5,5,25630,0,0,0,...,0,0,0,0,5,0,0,0,0,0
1,spotify:track:00GxbkrW4m1Tac5xySEJ4M,Ti volevo dedicare (feat. J-AX & Boomdabash),Rocco Hunt,Libertà,1,1,2980,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,spotify:track:00SmB7n85SKROGjybsyq5i,Bongo Bong,Manu Chao,Clandestino,13,6,1295035,0,0,0,...,0,0,0,0,1,4,0,0,1,7
3,spotify:track:00aZI3Ds8liGrcqEetsEYf,COMINCIA TU (feat. Rosa Chemical),Tananai,COMINCIA TU (feat. Rosa Chemical),1,1,4306,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,spotify:track:00bLSnliElWSOs1Ybt9dZm,The Monster,Eminem,The Marshall Mathers LP2,1,1,1182,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [157]:
# Sort the final grouped dataframe by 'listened' in descending order

df_grouped = df_grouped.sort_values(by='listened', ascending=False) 
df_grouped

,spotify_track_uri,title,artist,album,listened,skipped,ms_played,incognito_mode,offline,reason_start_trackerror,...,reason_end_unexpected-exit-while-paused,reason_end_logout,reason_end_trackerror,reason_end_unknown,reason_end_endplay,reason_end_fwdbtn,reason_end_remote,reason_end_unexpected-exit,reason_end_backbtn,reason_end_trackdone
2404,spotify:track:74tLlkN3rgVzRqQJgPfink,Money Trees,Kendrick Lamar,"good kid, m.A.A.d city",255,89,61333676,0,1,0,...,2,40,0,1,35,44,2,0,10,121
1881,spotify:track:5TRPicyLGbAF2LGBFbHGvO,Flashing Lights,Kanye West,Graduation,231,77,38365745,1,0,0,...,1,23,0,0,37,35,0,0,5,130
1100,spotify:track:3DK6m7It6Pw857FcQftMds,Runaway,Kanye West,My Beautiful Dark Twisted Fantasy,230,90,71652450,0,0,0,...,2,38,0,2,35,38,1,1,17,96
1084,spotify:track:3A4cpTBPaIQdtPFb5JxtaX,Slow Jamz,Twista,The College Dropout,217,75,45195568,1,1,0,...,0,24,0,0,38,31,0,0,6,118
279,spotify:track:0mEdbdeRFQwBhN4xfyIeUM,Can't Tell Me Nothing,Kanye West,Graduation,207,77,35514375,0,1,0,...,0,18,0,0,39,33,0,0,5,112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2612,spotify:track:7tOcPDj3vyopZ404pY6UuP,Shooting Stars,Bag Raiders,Shooting Stars,1,1,3305,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2611,spotify:track:7tI8dRuH2Yc6RuoTjxo4dU,Who,Jimin,MUSE,1,1,10030,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2610,spotify:track:7tEoJKbYdIHBfn7tTIyjHW,Drug Ballad,Eminem,The Marshall Mathers LP,1,0,300266,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1158,spotify:track:3MGuubCU1L7XWcTWc5Obb1,Hands,The Ting Tings,Hands,1,1,1430,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [158]:
# Export the final dataframe to a CSV file

df_grouped.to_csv('spotify_wrapped_2024.csv', index=False)  # data grouped by song
df.to_csv('spotify_wrapped_2024_extended.csv', index=False) # extended data with all plays

### Further Exploration

In [13]:
# Group by artist and calculate total ms_played and number of tracks played
artist_stats = df.groupby('master_metadata_album_artist_name').agg({
    'ms_played': 'sum',
    'spotify_track_uri': 'count'
}).reset_index()

# Rename columns for clarity
artist_stats.rename(columns={'ms_played': 'total_ms_played', 'spotify_track_uri': 'track_count'}, inplace=True)

# Sort by total_ms_played to find the most listened artists based on ms_played
most_listened_artists_ms_played = artist_stats.sort_values(by='total_ms_played', ascending=False)

# Sort by track_count to find the most listened artists based on number of tracks played
most_listened_artists_track_count = artist_stats.sort_values(by='track_count', ascending=False)

# Display the top 10 artists based on ms_played
print("Top 10 artists based on ms_played:")
print(most_listened_artists_ms_played.head(20))

# Display the top 10 artists based on number of tracks played
print("Top 10 artists based on number of tracks played:")
print(most_listened_artists_track_count.head(20))

Top 10 artists based on ms_played:
    master_metadata_album_artist_name  total_ms_played  track_count
404                        Kanye West        407804689         3153
408                    Kendrick Lamar        356952957         2429
239                            Eminem        232647674         2458
3                                2Pac         73536817          551
356                           J. Cole         56107056          459
806                            Twista         45195568          217
357                             JAY-Z         36014821          353
318                          Gorillaz         28473484          252
1                           21 Savage         24963577          182
162                          Coldplay         22954707          227
279                     Frah Quintale         21624661          155
216                           Dr. Dre         18679615          284
102                   Black Eyed Peas         16739441          141
133          

In [14]:
num_artists = artist_stats['master_metadata_album_artist_name'].nunique()
print(f"Number of unique artists: {num_artists}")

Number of unique artists: 866
